In [79]:
!pip install openai

In [80]:
from openai import OpenAI
from pathlib import Path
import getpass
import json

In [81]:
api_key = getpass.getpass("请输入 DeepSeek API Key：")

client = OpenAI(
    api_key=api_key,
    base_url="https://api.deepseek.com"
)

请输入 DeepSeek API Key：··········


In [82]:
WORKSPACE = Path("./workspace")
WORKSPACE.mkdir(exist_ok=True)

data_file = WORKSPACE / "data.txt"

data_file.write_text(
    "10\n20\n35\n17",
    encoding="utf-8"
)

print(data_file.read_text())

10
20
35
17


In [83]:
def read_file(path):
    file_path = WORKSPACE / path
    return file_path.read_text(encoding="utf-8")


def write_file(path, content):
    file_path = WORKSPACE / path
    file_path.write_text(content, encoding="utf-8")

    return f"成功写入 {path}"

In [84]:
state = {
    "task": "读取 data.txt 中的数字，计算它们的总和，并写入 result.txt",
    "step": 0,
    "status": "running",
    "conversation": [],
    "observations": [],
    "artifacts": []
}

In [85]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "read_file",
            "description": "读取 workspace 中指定文件的内容",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {
                        "type": "string",
                        "description": "文件名，例如 data.txt"
                    }
                },
                "required": ["path"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "write_file",
            "description": "向 workspace 中指定文件写入内容",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {
                        "type": "string"
                    },
                    "content": {
                        "type": "string"
                    }
                },
                "required": ["path", "content"]
            }
        }
    }
]

In [86]:
def build_context(state):

    messages = [
        {
            "role": "system",
            "content": (
                "你是一个文件处理 Agent。"
                "必须使用提供的工具完成任务。"
                "不要猜测文件内容。"
                "如果需要知道文件内容，必须先读取文件。"
            )
        },
        {
            "role": "user",
            "content": state["task"]
        }
    ]

    messages.extend(state["conversation"])

    return messages

In [87]:
def call_llm(context):

    response = client.chat.completions.create(
        model="deepseek-v4-pro",
        messages=context,
        tools=tools
    )

    return response.choices[0].message

In [88]:
context = build_context(state)

message = call_llm(context)

print("content:")
print(message.content)

print("\ntool_calls:")
print(message.tool_calls)

content:


tool_calls:
[ChatCompletionMessageFunctionToolCall(id='call_00_x5e6dKzh3NqBmXXTojWk1909', function=Function(arguments='{"path": "data.txt"}', name='read_file'), type='function', index=0)]


In [89]:
tool_call = message.tool_calls[0]

action = {
    "name": tool_call.function.name,
    "arguments": json.loads(
        tool_call.function.arguments
    )
}

print(action)

{'name': 'read_file', 'arguments': {'path': 'data.txt'}}


In [90]:
TOOL_REGISTRY = {
    "read_file": read_file,
    "write_file": write_file
}

In [91]:
def execute_action(action):

    name = action["name"]
    arguments = action["arguments"]

    if name not in TOOL_REGISTRY:
        raise ValueError(f"未知工具: {name}")

    tool = TOOL_REGISTRY[name]

    return tool(**arguments)

In [92]:
raw_result = execute_action(action)

print(raw_result)

10
20
35
17


In [93]:
def build_observation(action, raw_result):

    return {
        "action": action["name"],
        "success": True,
        "content": raw_result
    }

In [94]:
observation = build_observation(
    action,
    raw_result
)

print(observation)

{'action': 'read_file', 'success': True, 'content': '10\n20\n35\n17'}


In [95]:
state["observations"].append(observation)

state["step"] += 1

In [96]:
state["conversation"].append(
    message.model_dump()
)

In [97]:
state["conversation"].append(
    {
        "role": "tool",
        "tool_call_id": tool_call.id,
        "content": str(raw_result)
    }
)

In [98]:
print(state)

{'task': '读取 data.txt 中的数字，计算它们的总和，并写入 result.txt', 'step': 1, 'status': 'running', 'conversation': [{'content': '', 'refusal': None, 'role': 'assistant', 'annotations': None, 'audio': None, 'function_call': None, 'tool_calls': [{'id': 'call_00_x5e6dKzh3NqBmXXTojWk1909', 'function': {'arguments': '{"path": "data.txt"}', 'name': 'read_file'}, 'type': 'function', 'index': 0}], 'reasoning_content': 'We need to read data.txt, compute sum of numbers, write to result.txt. Need use tools. First read file.'}, {'role': 'tool', 'tool_call_id': 'call_00_x5e6dKzh3NqBmXXTojWk1909', 'content': '10\n20\n35\n17'}], 'observations': [{'action': 'read_file', 'success': True, 'content': '10\n20\n35\n17'}], 'artifacts': []}


In [99]:
context2 = build_context(state)

for item in context2:
    print(item)

{'role': 'system', 'content': '你是一个文件处理 Agent。必须使用提供的工具完成任务。不要猜测文件内容。如果需要知道文件内容，必须先读取文件。'}
{'role': 'user', 'content': '读取 data.txt 中的数字，计算它们的总和，并写入 result.txt'}
{'content': '', 'refusal': None, 'role': 'assistant', 'annotations': None, 'audio': None, 'function_call': None, 'tool_calls': [{'id': 'call_00_x5e6dKzh3NqBmXXTojWk1909', 'function': {'arguments': '{"path": "data.txt"}', 'name': 'read_file'}, 'type': 'function', 'index': 0}], 'reasoning_content': 'We need to read data.txt, compute sum of numbers, write to result.txt. Need use tools. First read file.'}
{'role': 'tool', 'tool_call_id': 'call_00_x5e6dKzh3NqBmXXTojWk1909', 'content': '10\n20\n35\n17'}


In [100]:
message2 = call_llm(context2)

print("content:")
print(message2.content)

print("\ntool_calls:")
print(message2.tool_calls)

content:


tool_calls:
[ChatCompletionMessageFunctionToolCall(id='call_00_IzSA4ybfshKengsykizX5449', function=Function(arguments='{"path": "result.txt", "content": "82"}', name='write_file'), type='function', index=0)]


In [101]:
tool_call2 = message2.tool_calls[0]

action2 = {
    "name": tool_call2.function.name,
    "arguments": json.loads(
        tool_call2.function.arguments
    )
}

print(action2)

{'name': 'write_file', 'arguments': {'path': 'result.txt', 'content': '82'}}


In [102]:
raw_result2 = execute_action(action2)

print(raw_result2)

成功写入 result.txt


In [103]:
observation2 = build_observation(
    action2,
    raw_result2
)

print(observation2)

{'action': 'write_file', 'success': True, 'content': '成功写入 result.txt'}


In [104]:
state["conversation"].append(
    message2.model_dump()
)

In [105]:
state["conversation"].append(
    {
        "role": "tool",
        "tool_call_id": tool_call2.id,
        "content": str(raw_result2)
    }
)

In [106]:
state["observations"].append(observation2)

state["artifacts"].append(
    action2["arguments"]["path"]
)

state["step"] += 1

In [107]:
print(state["step"])
print(state["artifacts"])
print(state["observations"])

2
['result.txt']
[{'action': 'read_file', 'success': True, 'content': '10\n20\n35\n17'}, {'action': 'write_file', 'success': True, 'content': '成功写入 result.txt'}]


In [108]:
def verify(state):

    result_file = WORKSPACE / "result.txt"

    if not result_file.exists():
        return False

    result = result_file.read_text(
        encoding="utf-8"
    ).strip()

    return result == "82"

In [109]:
success = verify(state)

print("Verify:", success)

Verify: True


In [110]:
print(
    (WORKSPACE / "result.txt").read_text(
        encoding="utf-8"
    )
)

82


In [111]:
# 删除上一次实验生成的 result.txt
result_file = WORKSPACE / "result.txt"

if result_file.exists():
    result_file.unlink()


def create_initial_state():
    return {
        "task": "读取 data.txt 中的数字，计算它们的总和，并写入 result.txt",
        "step": 0,
        "status": "running",

        # 保存与 LLM 的多轮交互
        "conversation": [],

        # Harness 保存的 observation
        "observations": [],

        # 任务产生的 artifact
        "artifacts": []
    }


state = create_initial_state()

In [112]:
def verify(state):

    result_file = WORKSPACE / "result.txt"

    if not result_file.exists():
        return False

    content = result_file.read_text(
        encoding="utf-8"
    ).strip()

    return content == "82"

In [113]:
def run_agent(state, max_steps=10):

    while state["status"] == "running":

        print("\n" + "=" * 60)
        print(f"第 {state['step']} 轮")
        print("=" * 60)

        # 防止无限循环
        if state["step"] >= max_steps:
            state["status"] = "failed"
            print("超过最大执行轮数，停止任务")
            break


        # ==================================================
        # 1. Context Manager
        # ==================================================

        context = build_context(state)

        print("\n[Context]")
        for msg in context:
            print(msg)


        # ==================================================
        # 2. LLM
        # ==================================================

        message = call_llm(context)

        print("\n[LLM Output]")
        print("content:", message.content)
        print("tool_calls:", message.tool_calls)


        # 把模型本轮输出写入 conversation
        state["conversation"].append(
            message.model_dump()
        )


        # ==================================================
        # 3. 判断 Model Output 类型
        # ==================================================

        # 情况 A：LLM 要调用工具
        if message.tool_calls:

            for tool_call in message.tool_calls:

                # ------------------------------------------
                # 4. Model Output -> Action
                # ------------------------------------------

                action = {
                    "name": tool_call.function.name,
                    "arguments": json.loads(
                        tool_call.function.arguments
                    )
                }

                print("\n[Action]")
                print(action)


                # ------------------------------------------
                # 5. Action Interface
                # ------------------------------------------

                try:
                    raw_result = execute_action(action)

                    success = True

                except Exception as e:
                    raw_result = str(e)

                    success = False


                print("\n[Environment Raw Result]")
                print(raw_result)


                # ------------------------------------------
                # 6. Observation Interface
                # ------------------------------------------

                observation = {
                    "action": action["name"],
                    "success": success,
                    "content": raw_result
                }

                print("\n[Observation]")
                print(observation)


                # ------------------------------------------
                # 7. Update State
                # ------------------------------------------

                state["observations"].append(
                    observation
                )

                if (
                    success
                    and action["name"] == "write_file"
                ):
                    path = action["arguments"]["path"]

                    if path not in state["artifacts"]:
                        state["artifacts"].append(path)


                # ------------------------------------------
                # 8. 把 Tool Result 返回给 LLM 历史
                # ------------------------------------------

                state["conversation"].append(
                    {
                        "role": "tool",
                        "tool_call_id": tool_call.id,
                        "content": str(raw_result)
                    }
                )


            # 一轮 tool call 完成
            state["step"] += 1


            # ------------------------------------------
            # 9. Verification
            # ------------------------------------------

            verified = verify(state)

            print("\n[Verify]")
            print(verified)

            if verified:

                state["status"] = "completed"

                print("\n任务验证成功")
                break


            # verify 不通过
            # while 自动进入下一轮
            continue


        # ==================================================
        # 情况 B：LLM 没有 Tool Call，只返回普通内容
        # ==================================================

        else:

            print("\n[Assistant Message]")
            print(message.content)

            verified = verify(state)

            print("\n[Verify]")
            print(verified)

            if verified:
                state["status"] = "completed"
                print("\n任务完成")
                break

            else:
                # 模型以为自己做完了，但 Harness 不认可
                state["conversation"].append(
                    {
                        "role": "user",
                        "content": (
                            "系统验证发现任务尚未真正完成。"
                            "请继续使用工具完成任务，"
                            "不要仅用文字声明任务完成。"
                        )
                    }
                )

                state["step"] += 1

    return state

In [114]:
final_state = run_agent(state)


第 0 轮

[Context]
{'role': 'system', 'content': '你是一个文件处理 Agent。必须使用提供的工具完成任务。不要猜测文件内容。如果需要知道文件内容，必须先读取文件。'}
{'role': 'user', 'content': '读取 data.txt 中的数字，计算它们的总和，并写入 result.txt'}

[LLM Output]
content: 
tool_calls: [ChatCompletionMessageFunctionToolCall(id='call_00_NvRUYczfpGLlYhR7jIpS7433', function=Function(arguments='{"path": "data.txt"}', name='read_file'), type='function', index=0)]

[Action]
{'name': 'read_file', 'arguments': {'path': 'data.txt'}}

[Environment Raw Result]
10
20
35
17

[Observation]
{'action': 'read_file', 'success': True, 'content': '10\n20\n35\n17'}

[Verify]
False

第 1 轮

[Context]
{'role': 'system', 'content': '你是一个文件处理 Agent。必须使用提供的工具完成任务。不要猜测文件内容。如果需要知道文件内容，必须先读取文件。'}
{'role': 'user', 'content': '读取 data.txt 中的数字，计算它们的总和，并写入 result.txt'}
{'content': '', 'refusal': None, 'role': 'assistant', 'annotations': None, 'audio': None, 'function_call': None, 'tool_calls': [{'id': 'call_00_NvRUYczfpGLlYhR7jIpS7433', 'function': {'arguments': '{"path": "data.txt"}', '

In [115]:
!find /content -name "*.ipynb"